# APIs and streaming

## Learning goals
- Contrast the offline engine API with the OpenAI-style HTTP API
- Recognize the real media endpoints vLLM-Omni exposes
- Simulate a streamed response and reason about TTFT

> **Mac track.** Every executed cell below is a pure-Python simulation that runs on any Mac with no GPU. Cells labelled **Linux GPU lab** show the *real* vLLM-Omni commands — they are shown as text and are **not** run here, because the official `vllm serve --omni` runtime is CUDA-only.

## Two ways in

- **Offline**: construct an engine in-process and call it with a batch of prompts (great for benchmarking and pipelines).
- **Online**: an OpenAI-compatible HTTP server with familiar endpoints, so existing clients work with minimal changes.

vLLM-Omni adds media endpoints on top of chat/completions, e.g. `/v1/audio/speech` and image/video generation routes.

In [ ]:
endpoints = {
    '/v1/chat/completions': 'text + multimodal input -> text (+ optional audio)',
    '/v1/audio/speech':     'text -> audio (TTS path: Talker + Code2Wav)',
    '/v1/images/generations': 'text -> image (DiT stage)',
}
for route, desc in endpoints.items():
    print(f'{route:26} {desc}')

## Simulating a streamed chat response

Streaming yields tokens as they are produced. The first token's arrival is the time-to-first-token (TTFT); the gap between tokens is the inter-token latency.

In [ ]:
def stream(tokens, ttft, itl):
    t = ttft
    for tok in tokens:
        yield round(t, 2), tok
        t += itl
for arrival, tok in stream(['Hello', 'from', 'vLLM', '-', 'Omni'], ttft=0.30, itl=0.05):
    print(f't={arrival:4.2f}s  {tok}')

## Exercise

A client shows text and speaks it. With streaming, should it wait for the whole response before starting TTS, or pipe tokens through as they arrive? Justify in terms of TTFT.

In [ ]:
# Solution: pipe them through. Streaming to the Talker/TTS as tokens arrive means audio can
# start near the text TTFT instead of after the full text completes -- the same decoupling
# the orchestrator uses between stages (notebook 06).
print('Stream through: perceived latency ~ text TTFT, not full-response time.')

## Source lab

Read the OpenAI-compatible server entrypoints in `vllm_omni/entrypoints/`. Match each media endpoint to the stage path it drives.